# Imma Hungry (CIS 1921 Project)

Harry Li, Nabo Yu

We want to build an optimizer for the [Share Food Program](https://www.sharefoodprogram.org/philly-food-rescue).

In brief, the problem is as follows. We have a network of volunteer drivers to pick up perishable food from grocers, restaurants, and caterers, and they deliver them to affordable housing communities and nonprofits.

### Data Source

We use real world data from [Datasets - OpenDataPhilly](https://opendataphilly.org/datasets/?category=food), including [OpenMaps - OpenDataPhilly](https://opendataphilly.org/datasets/openmaps/) and [Free Food & Meal Sites - OpenDataPhilly](https://opendataphilly.org/datasets/free-meals/), and Google Maps API, including the [Compute Route Matrix](https://developers.google.com/maps/documentation/routes/compute-route-matrix-over) API.

- The OpenMaps dataset provides the geographic information of Philly.
- The Free Food & Meal Sites dataset provides the food donations information in Philly.
- The Compute Route Matrix API provides the travel route matrix (i.e., generalized distance matrix taking route and traffic into consideration) between the locations.

### Formulation

Let there be $X$ volunteer drivers with carrying capacity $c$ scattered around Philadelphia, $Y$ active food donors with $s$ hours of spoilage windows, and $Z$ drop-off locations with pantry capacity $p$. We will optimize the driver-donation assignments and the route for each assignment by minimizing *food waste* and maximizing *nutrient availability* in the process.

### Evaluation

We evaluate the system by running simulations of sampled location subsets in the OpenDataPhilly dataset.

1. With small-scale driver and location samples to verify our pipeline works.
2. With larger-scale driver and location samples to compare our solver with
   - a baseline greedy strategy, and
   - some heuristic strategy
   
   to see if our solver does indeed improve upon naive approaches.

In [ ]:
import polars as pl

print("OK!")

## Data Source

In [ ]:
def fetch_data(remote: str, local: str):
    import os

    if os.path.exists(local):
        return
    os.makedirs(os.path.dirname(local), exist_ok=True)
    import urllib.request

    urllib.request.urlretrieve(remote, local)

There is no public information on food donors. We use the information of restaurants instead, assuming they are willing to donate food.

In [ ]:
# https://opendataphilly.org/datasets/licenses-and-inspections-business-licenses/
fetch_data(
    "https://phl.carto.com/api/v2/sql?q=SELECT+*,+ST_Y(the_geom)+AS+lat,+ST_X(the_geom)+AS+lng+FROM+business_licenses&filename=business_licenses&format=csv&skipfields=cartodb_id",
    "data/business_licenses.csv",
)

donor_locations = pl.read_csv("data/business_licenses.csv", infer_schema_length=65536)

donor_locations = (
    donor_locations.filter(
        pl.col("licensestatus") == "Active",
        pl.col("licensetype").is_in(
            [
                "Food Preparing and Serving",
                "Food Preparing and Serving (30+)",
                "Food Establishment, Retail Permanent",
            ]
        ),
    )
    .select(
        pl.col("business_name"),
        pl.col("address"),
        pl.col("lat"),
        pl.col("lng"),
    )
    .drop_nulls(subset=["lat", "lng"])
)

donor_locations = donor_locations.sample(32, seed=1921)
donor_locations

We use the information of existing share food program sites.

In [ ]:
# https://opendataphilly.org/datasets/free-meals/
# spatialRefId=4326 gives WGS84 lat/lng directly in x (lng) and y (lat) columns
fetch_data(
    "https://hub.arcgis.com/api/v3/datasets/5825a32bb8844bb097f7a16d4fbf4f23_0/downloads/data?format=csv&spatialRefId=4326&where=1%3D1",
    "data/free_meal_sites.csv",
)

dropoff_locations = pl.read_csv("data/free_meal_sites.csv")

dropoff_locations = (
    dropoff_locations.filter(pl.col("category").str.contains("SHARE FOOD PROGRAM"))
    .select(
        pl.col("site_name"),
        pl.col("address"),
        pl.col("x").alias("lng"),
        pl.col("y").alias("lat"),
    )
    .drop_nulls(subset=["lat", "lng"])
)

dropoff_locations = dropoff_locations.sample(32, seed=1921)
dropoff_locations

We use the [OSRM Table API](http://router.project-osrm.org) (free, no key, OpenStreetMap-based) to compute driving distances and durations for all (donor + drop-off) → drop-off pairs in a single request.

Before querying, we filter pairs by **crow-flies (Haversine) distance**: pairs exceeding `MAX_CROW_FLIES_KM` are excluded from the output and treated as infeasible by the optimizer. This doesn't reduce OSRM calls (the table service always computes the full matrix), but it shrinks the optimizer's search space. If you later switch to the Google Maps Route Matrix API, the same filter will also skip billable elements.

**On duration and time-of-day**: OSRM uses static average speeds (no live traffic). For a planning/simulation context this is fine. If you need traffic-aware durations, switch to the Google Maps Routes API — the $200/month free credit covers ~10,000 route elements and should be sufficient for the project.

In [ ]:
import os
import numpy as np

# Set to "osrm" for free routing (no API key, uses OpenStreetMap data, no live traffic)
# Set to "google" for traffic-aware routing (requires GOOGLE_MAPS_API_KEY env var or ~/.gcp_api_key)
ROUTING_BACKEND = "osrm"


def _load_google_api_key():
    key = os.environ.get("GOOGLE_MAPS_API_KEY", "")
    if not key:
        keyfile = os.path.expanduser("~/.gcp_api_key")
        if os.path.exists(keyfile):
            with open(keyfile) as f:
                key = f.read().strip()
    return key


GOOGLE_API_KEY = _load_google_api_key()

# Pairs with crow-flies distance beyond this are treated as infeasible by the optimizer.
# ~10 km keeps routes within a reasonable volunteer driving range in Philly (~6 miles).
# For the Google backend this also skips those elements from the API request (saves cost).
MAX_CROW_FLIES_KM = 10.0

# Sentinel used in distance/duration matrices for non-viable or unreachable pairs.
# Large enough that no optimizer will select these edges; small enough to avoid int32 overflow when summed.
INFEASIBLE = 10**7  # ~10,000 km / ~115 days — clearly unreachable


def haversine_matrix(lat1, lng1, lat2, lng2):
    """Returns (m, n) matrix of crow-flies distances in km."""
    lat1 = np.radians(np.asarray(lat1, dtype=float))[:, None]
    lng1 = np.radians(np.asarray(lng1, dtype=float))[:, None]
    lat2 = np.radians(np.asarray(lat2, dtype=float))[None, :]
    lng2 = np.radians(np.asarray(lng2, dtype=float))[None, :]
    dlat, dlng = lat2 - lat1, lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a.clip(0, 1)))


def _fetch_osrm(n_donors, n_dropoffs, all_lat, all_lng):
    """Returns (durations_s, distances_m) as (n_total, n_dropoffs) float arrays. NaN = unreachable."""
    import json, urllib.request

    n_total = n_donors + n_dropoffs
    coords = ";".join(f"{lng},{lat}" for lat, lng in zip(all_lat, all_lng))
    src_idx = ";".join(str(i) for i in range(n_total))
    dst_idx = ";".join(str(n_donors + j) for j in range(n_dropoffs))
    url = (
        f"http://router.project-osrm.org/table/v1/driving/{coords}"
        f"?sources={src_idx}&destinations={dst_idx}"
        f"&annotations=duration,distance&generate_hints=false"
    )
    with urllib.request.urlopen(url, timeout=60) as resp:
        data = json.loads(resp.read())
    return (
        np.array(data["durations"], dtype=float),  # (n_total, n_dropoffs)
        np.array(data["distances"], dtype=float),  # (n_total, n_dropoffs)
    )


def _fetch_google(viable, n_donors, n_dropoffs, all_addresses):
    """Returns (durations_s, distances_m) as (n_total, n_dropoffs) float arrays. NaN = unreachable."""
    from google.api_core.client_options import ClientOptions
    import google.maps.routing_v2 as routes_api

    client = routes_api.RoutesClient(
        client_options=ClientOptions(api_key=GOOGLE_API_KEY)
    )
    n_total = n_donors + n_dropoffs

    all_origins = [
        routes_api.RouteMatrixOrigin(
            waypoint=routes_api.Waypoint(address=all_addresses[i])
        )
        for i in range(n_total)
    ]
    all_destinations = [
        routes_api.RouteMatrixDestination(
            waypoint=routes_api.Waypoint(address=all_addresses[n_donors + j])
        )
        for j in range(n_dropoffs)
    ]

    # Skip origins with no viable destinations — saves billable API elements
    viable_origins = [i for i in range(n_total) if viable[i].any()]
    if skipped := n_total - len(viable_origins):
        print(f"  Skipping {skipped} origins with no viable destinations")

    durations = np.full((n_total, n_dropoffs), np.nan)
    distances = np.full((n_total, n_dropoffs), np.nan)

    BATCH_SIZE = 50 - n_dropoffs  # origins + destinations ≤ 50 per request
    for start in range(0, len(viable_origins), BATCH_SIZE):
        batch_indices = viable_origins[start : start + BATCH_SIZE]
        response = client.compute_route_matrix(
            routes_api.ComputeRouteMatrixRequest(
                origins=[all_origins[i] for i in batch_indices],
                destinations=all_destinations,
            ),
            metadata=[
                (
                    "x-goog-fieldmask",
                    "originIndex,destinationIndex,duration,distanceMeters,status",
                )
            ],
        )
        for el in response:
            i, j = batch_indices[el.origin_index], el.destination_index
            distances[i, j] = el.distance_meters
            durations[i, j] = el.duration.seconds

    return durations, distances


def fetch_matrix():
    CACHE_PATH = f"data/route_matrix_{ROUTING_BACKEND}.npz"
    if os.path.exists(CACHE_PATH):
        data = np.load(CACHE_PATH)
        return {
            "locations": data["locations"].tolist(),
            "n_donors": int(data["n_donors"]),
            "distance": data["distance"],
            "duration": data["duration"],
        }

    n_donors = len(donor_locations)
    n_dropoffs = len(dropoff_locations)

    all_lat = donor_locations["lat"].to_list() + dropoff_locations["lat"].to_list()
    all_lng = donor_locations["lng"].to_list() + dropoff_locations["lng"].to_list()
    locations = (
        donor_locations["business_name"].to_list()
        + dropoff_locations["site_name"].to_list()
    )

    # Crow-flies filter — only road-distance/duration for nearby pairs
    crow_km = haversine_matrix(
        all_lat,
        all_lng,
        dropoff_locations["lat"].to_list(),
        dropoff_locations["lng"].to_list(),
    )  # (n_total, n_dropoffs)
    viable = crow_km <= MAX_CROW_FLIES_KM
    n_total = n_donors + n_dropoffs
    print(
        f"Crow-flies filter ({ROUTING_BACKEND}): {viable.sum()} / {n_total * n_dropoffs} pairs viable"
    )

    if ROUTING_BACKEND == "osrm":
        raw_dur, raw_dist = _fetch_osrm(n_donors, n_dropoffs, all_lat, all_lng)
    elif ROUTING_BACKEND == "google":
        if not GOOGLE_API_KEY:
            raise ValueError(
                "No API key found — set GOOGLE_MAPS_API_KEY env var or write it to ~/.gcp_api_key"
            )
        all_addresses = [
            f"{a}, Philadelphia, PA" for a in donor_locations["address"].to_list()
        ] + [f"{a}, Philadelphia, PA" for a in dropoff_locations["address"].to_list()]
        raw_dur, raw_dist = _fetch_google(viable, n_donors, n_dropoffs, all_addresses)
    else:
        raise ValueError(
            f"Unknown ROUTING_BACKEND {ROUTING_BACKEND!r} — use 'osrm' or 'google'"
        )

    # Merge viable mask with reachability; fill everything else with INFEASIBLE sentinel
    ok = viable & ~np.isnan(raw_dist) & ~np.isnan(raw_dur)
    distance = np.where(ok, raw_dist, INFEASIBLE).astype(np.int32)
    duration = np.where(ok, raw_dur, INFEASIBLE).astype(np.int32)

    os.makedirs("data", exist_ok=True)
    np.savez_compressed(
        CACHE_PATH,
        locations=np.array(locations),
        n_donors=np.array([n_donors]),
        distance=distance,
        duration=duration,
    )
    return {
        "locations": locations,
        "n_donors": n_donors,
        "distance": distance,
        "duration": duration,
    }


# route_matrix["locations"][i]  — name of location i
# route_matrix["n_donors"]      — locations[:n_donors] are donors; the rest are dropoffs
# route_matrix["distance"][i,j] — road distance (meters) from location i to dropoff (n_donors+j)
# route_matrix["duration"][i,j] — driving time (seconds); INFEASIBLE sentinel where not viable
route_matrix = fetch_matrix()

nd, n = route_matrix["n_donors"], len(route_matrix["locations"])
viable_count = (route_matrix["distance"] < INFEASIBLE).sum()
print(f"{nd} donors + {n - nd} dropoffs | {viable_count} viable pairs")
route_matrix["distance"]